# RAG From Scratch: Multimodal Document RAG

PDFs and slide decks often encode meaning in layout, tables, figures, and screenshots. Text-only chunking loses that information. This notebook shows the shape of a page-level visual document RAG pipeline and includes an optional Gemini multimodal embedding sketch.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pages = [
    {
        "doc_id": "report.pdf",
        "page": 1,
        "text": "Revenue increased in Q4. The chart shows enterprise growth.",
        "image_path": "data/report_page_1.png",
    },
    {
        "doc_id": "report.pdf",
        "page": 2,
        "text": "The table lists retrieval latency and answer quality for RAG variants.",
        "image_path": "data/report_page_2.png",
    },
]

texts = [page["text"] for page in pages]
vectorizer = TfidfVectorizer().fit(texts)
page_matrix = vectorizer.transform(texts)

def retrieve_pages(query, k=1):
    scores = cosine_similarity(vectorizer.transform([query]), page_matrix).ravel()
    ranked = sorted(zip(scores, pages), reverse=True, key=lambda item: item[0])
    return [page for _, page in ranked[:k]]

retrieve_pages("Which page compares RAG latency?")

## Page citation payload

For visual documents, keep page number, image path, OCR text, and region metadata together. Generation can then cite exact pages or regions instead of vague chunks.

In [ ]:
def citation_payload(page):
    return {
        "source": page["doc_id"],
        "page": page["page"],
        "image_path": page["image_path"],
        "snippet": page["text"][:160],
    }

[citation_payload(page) for page in retrieve_pages("RAG latency table")]

In [ ]:
# Optional Gemini multimodal embedding sketch. Requires GOOGLE_API_KEY and real image files.
# from pathlib import Path
# from google import genai
# from google.genai import types
#
# client = genai.Client()
# image_bytes = Path("data/report_page_2.png").read_bytes()
# result = client.models.embed_content(
#     model="gemini-embedding-2",
#     contents=[
#         "A report page containing a table about RAG latency and answer quality.",
#         types.Part.from_bytes(data=image_bytes, mime_type="image/png"),
#     ],
# )
# result.embeddings[0].values[:5]